# Aim 1 sensitivity S4 (ICD-coded social needs) and E-values

Run **after** `03_aim1_s4_flags.sql` has created `ctheg-scores.ins_icu.aim1_s4_flags`.
Uses the same cohort, variable preparation and model functions as `aim1_models_FINAL.ipynb`
(two-part days model with a post-ICU ward stay defined as > 4 h).

- **S4:** primary covariates + three ICD-coded proxies (housing/economic circumstances, substance use
  disorder, serious mental illness). Exploratory: codes are discharge-assigned proxies, not verified
  baseline social needs, and could partly reflect the hospitalization itself.
- **E-values** (VanderWeele & Ding 2017) for the adjusted Medicaid/private risk ratio of each destination,
  primary and S4 models: the minimum strength of association (risk-ratio scale) an unmeasured confounder
  would need with both insurance and the outcome to explain away the estimate (point) or move the CI to
  include 1 (CI bound). A benchmark, not proof that residual confounding is absent.
- Paired bootstrap: each replicate refits the primary and S4 models on the same resampled patients.

Privacy: row-level data stay in this session. Clear outputs before saving/sharing; share only the results file.

In [ ]:
# ---- Settings ---------------------------------------------------------------
PROJECT  = "ctheg-scores"
SEED     = 20261112
B_S4     = 200     # bootstrap reps (use 20 for a quick test)
OUTFILE  = "aim1_s4_results.xlsx"
MIN_CELL = 11      # aggregate counts below this are suppressed in exported tables

import time, warnings
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
import patsy

from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # convergence is checked explicitly below

class NotConverged(Exception):
    pass

def check(res, label):
    ok = res.mle_retvals.get("converged", True) if hasattr(res, "mle_retvals") and res.mle_retvals else getattr(res, "converged", True)
    if not ok:
        raise NotConverged(label)
    return res
pd.set_option("display.width", 200, "display.max_columns", 50)
print("statsmodels", statsmodels.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
from google.colab import auth
auth.authenticate_user()
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)
COLS = """v.ins_cat, v.dc_group, v.post_icu_days, v.any_post_icu_stay, v.dc_ama, v.dc_acute_transfer,
          v.time_valid_strict, v.age_at_admit, v.gender, v.race_cat, v.sofa_day1, v.cci_no_age,
          v.adm_type_cat, v.invasive_vent, v.vasopressor, v.icu_los_days_total, v.first_careunit,
          v.anchor_year_group, v.language_cat,
          CAST(s.housing_econ_icd AS INT64) AS housing, CAST(s.sud_icd AS INT64) AS sud,
          CAST(s.smi_icd AS INT64) AS smi"""
raw18 = client.query(f"""SELECT {COLS} FROM `{PROJECT}.ins_icu.aim1_primary_18_64` v
                         LEFT JOIN `{PROJECT}.ins_icu.aim1_s4_flags` s USING (hadm_id)""").to_dataframe()
assert raw18[["housing", "sud", "smi"]].notna().all().all(), "some cohort admissions lack S4 flags"
print("18-64:", raw18.shape)

In [ ]:
# ---- Variable preparation -----------------------------------------------------
DC_LEVELS = ["home", "home_health", "snf", "rehab", "ltach", "other"]   # code 0 = home (reference)

ICU_MAP = {
    "Cardiac Vascular Intensive Care Unit (CVICU)":       "Cardiac",
    "Coronary Care Unit (CCU)":                           "Cardiac",
    "Medical Intensive Care Unit (MICU)":                 "Medical",
    "Medical/Surgical Intensive Care Unit (MICU/SICU)":   "Mixed",
    "Surgical Intensive Care Unit (SICU)":                "Surgical/Trauma",
    "Trauma SICU (TSICU)":                                "Surgical/Trauma",
    "Neuro Surgical Intensive Care Unit (Neuro SICU)":    "Neuro",
    "Neuro Intermediate":                                 "Neuro",
    "Neuro Stepdown":                                     "Neuro",
}   # anything else (PACU, Medicine, 'Intensive Care Unit (ICU)', small surgical units; ~60 patients) -> 'Mixed'
#     (merged 2026-11-12: a separate tiny group caused non-convergence in bootstrap resamples)

REF = {"race_cat": "White", "adm_type_cat": "Emergency", "icu_type": "Medical",
       "anchor_year_group": "2008 - 2010", "language_cat": "English"}

def prep(raw, exposure_levels):
    d = raw[raw["ins_cat"].isin(exposure_levels)].copy()
    d["icu_type"]    = d["first_careunit"].map(ICU_MAP).fillna("Mixed")
    d["female"]      = (d["gender"] == "F").astype(int)
    d["age_c"]       = (d["age_at_admit"] - 50) / 10            # per 10 years, centred at 50
    d["log_icu_los"] = np.log1p(d["icu_los_days_total"].astype(float))
    d["vent"]        = d["invasive_vent"].astype(int)
    d["pressor"]     = d["vasopressor"].astype(int)
    d["any_stay"]    = d["any_post_icu_stay"].astype(int)
    d["days_total"]  = np.where(d["any_stay"] == 1, d["post_icu_days"].astype(float), 0.0)  # two-part: <=4 h counted as 0
    d["post_icu_days"] = d["post_icu_days"].astype(float)
    d["sofa_day1"]   = d["sofa_day1"].astype(float)
    d["cci_no_age"]  = d["cci_no_age"].astype(float)
    d["facility"]    = d["dc_group"].isin(["snf", "rehab", "ltach"]).astype(int)
    for c in ["housing", "sud", "smi"]:          # S4 flags travel with each row
        if c in d:
            d[c] = d[c].astype(int)
    for c in ["dc_ama", "dc_acute_transfer", "time_valid_strict"]:
        d[c] = d[c].astype(bool)
    d["dc_code"] = pd.Categorical(d["dc_group"], categories=DC_LEVELS).codes
    d["ins"] = pd.Categorical(d["ins_cat"], categories=exposure_levels)   # first level = reference
    for c, ref in REF.items():                                            # fixed level order, reference first
        levels = [ref] + sorted(x for x in d[c].dropna().unique() if x != ref)
        d[c] = pd.Categorical(d[c], categories=levels)
    n_missing = d[["sofa_day1", "cci_no_age", "dc_code"]].isna().sum()
    assert (d["dc_code"] >= 0).all(), "unexpected discharge group"
    print(f"n = {len(d):,} | missing:", n_missing.to_dict())
    return d.dropna(subset=["sofa_day1", "cci_no_age"]).reset_index(drop=True)

COV = ("age_c + I(age_c**2) + female + C(race_cat) + sofa_day1 + cci_no_age + C(adm_type_cat)"
       " + vent + pressor + log_icu_los + C(icu_type) + C(anchor_year_group)")
COV_LANG = COV + " + C(language_cat)"
EXPO = "C(ins)"

d18 = prep(raw18, ["Private", "Medicaid"])
d18["medicaid"] = (d18["ins_cat"] == "Medicaid").astype(int)
print(d18["ins"].value_counts())

In [ ]:
# ---- Helpers: counterfactual prediction, bootstrap, summaries ------------------
def set_exposure(d, level):
    dd = d.copy()
    dd["ins"] = pd.Categorical([level] * len(dd), categories=d["ins"].cat.categories)
    return dd

def bootstrap(fn, d, reps, seed=SEED, label=""):
    """Nonparametric bootstrap over patients (one row per patient). fn(d) -> dict of named estimates."""
    rng, out, nc, err, t0 = np.random.default_rng(seed), [], 0, 0, time.time()
    for b in range(reps):
        db = d.iloc[rng.integers(0, len(d), len(d))].reset_index(drop=True)
        try:
            out.append(fn(db))
        except NotConverged:
            nc += 1          # dropped; count reported
        except Exception:
            err += 1
    print(f"[{label}] bootstrap {len(out)}/{reps} used | non-converged dropped: {nc} | errors: {err} | {time.time() - t0:.0f}s")
    return pd.DataFrame(out)

def summarise(point, boot):
    rows = []
    for k, v in point.items():
        lo, hi = np.nanpercentile(boot[k], [2.5, 97.5]) if k in boot else (np.nan, np.nan)
        rows.append({"estimate": k, "value": v, "ci_low": lo, "ci_high": hi})
    return pd.DataFrame(rows)

def coef_table(res, exp=True, name=""):
    ci = res.conf_int()
    t = pd.DataFrame({"coef": res.params, "se": res.bse, "p": res.pvalues,
                      "ci_low": ci.iloc[:, 0], "ci_high": ci.iloc[:, 1]})
    if exp:
        t["ratio"], t["ratio_ci_low"], t["ratio_ci_high"] = np.exp(t["coef"]), np.exp(t["ci_low"]), np.exp(t["ci_high"])
    t.insert(0, "model", name)
    return t.reset_index().rename(columns={"index": "term"})

In [ ]:
# ---- Outcome 1: discharge destination (multinomial logit + g-computation) ------
def fit_mnlogit(d, cov=COV, start=None):
    res = smf.mnlogit(f"dc_code ~ {EXPO} + {cov}", d).fit(
        disp=0, maxiter=60, method="newton", start_params=start)
    return check(res, "mnlogit")

def dest_estimates(d, cov=COV, start=None):
    res = fit_mnlogit(d, cov, start)
    levels, ref = list(d["ins"].cat.categories), d["ins"].cat.categories[0]
    probs = {lvl: np.asarray(res.predict(set_exposure(d, lvl))).mean(axis=0) for lvl in levels}
    out = {}
    for lvl in levels:
        for j, dest in enumerate(DC_LEVELS):
            out[f"P({dest}) | {lvl}"] = probs[lvl][j]
    for lvl in levels[1:]:
        for j, dest in enumerate(DC_LEVELS):
            out[f"Diff P({dest}) | {lvl} - {ref}"] = probs[lvl][j] - probs[ref][j]
    return out, res

In [ ]:
# ---- Outcome 2: post-ICU hospital days (two-part model) ------------------------
def days_estimates(d, cov=COV):
    p1 = check(smf.logit(f"any_stay ~ {EXPO} + {cov}", d).fit(disp=0, maxiter=60), "logit part 1")
    pos = d[d["any_stay"] == 1]
    p2 = check(smf.glm(f"post_icu_days ~ {EXPO} + {cov}", pos,
                 family=sm.families.Gamma(sm.families.links.Log())).fit(), "gamma part 2")
    levels, ref = list(d["ins"].cat.categories), d["ins"].cat.categories[0]
    out, e = {}, {}
    for lvl in levels:
        dd = set_exposure(d, lvl)
        pa, cd = np.asarray(p1.predict(dd)), np.asarray(p2.predict(dd))
        e[lvl] = (pa.mean(), cd.mean(), (pa * cd).mean())
        out[f"P(any post-ICU stay) | {lvl}"]   = e[lvl][0]
        out[f"E(days | stay) | {lvl}"]         = e[lvl][1]
        out[f"E(days) overall | {lvl}"]        = e[lvl][2]
    for lvl in levels[1:]:
        out[f"Diff P(any stay) | {lvl} - {ref}"]   = e[lvl][0] - e[ref][0]
        out[f"Diff E(days | stay) | {lvl} - {ref}"] = e[lvl][1] - e[ref][1]
        out[f"Diff E(days) overall | {lvl} - {ref}"] = e[lvl][2] - e[ref][2]
        out[f"Ratio E(days | stay) | {lvl} / {ref}"] = float(np.exp(p2.params[f"C(ins)[T.{lvl}]"]))
    return out, (p1, p2)

In [ ]:
# ---- S4 flags: prevalence by insurance (aggregate; small cells suppressed) ----------
prev = (d18.groupby("ins_cat")[["housing", "sud", "smi"]].agg(["sum", "mean"]))
prev_tab = pd.DataFrame({f: [f"{int(prev.loc[g, (f, 'sum')]):,} ({100 * prev.loc[g, (f, 'mean')]:.1f}%)"
                              if prev.loc[g, (f, "sum")] >= MIN_CELL else f"<{MIN_CELL}"
                              for g in prev.index] for f in ["housing", "sud", "smi"]}, index=prev.index)
prev_tab

In [ ]:
# ---- Primary vs S4: paired estimates and E-values -----------------------------------
COV_S4 = COV + " + housing + sud + smi"

def evalue(rr):
    rr = max(rr, 1 / rr)
    return rr + np.sqrt(rr * (rr - 1))

def evalue_bound(lo, hi):
    if lo <= 1 <= hi:
        return 1.0
    return evalue(lo if lo > 1 else hi)

def s4_block(d):
    out = {}
    for label, cov in [("Primary", COV), ("S4", COV_S4)]:
        a, _ = dest_estimates(d, cov)
        b, _ = days_estimates(d, cov)
        for dest in DC_LEVELS:
            pM, pP = a[f"P({dest}) | Medicaid"], a[f"P({dest}) | Private"]
            out[f"{label} | Diff P({dest})"] = pM - pP
            out[f"{label} | RR({dest})"] = pM / pP
        out[f"{label} | Diff P(any stay)"] = b["Diff P(any stay) | Medicaid - Private"]
        out[f"{label} | Diff E(days | stay)"] = b["Diff E(days | stay) | Medicaid - Private"]
        out[f"{label} | Ratio E(days | stay)"] = b["Ratio E(days | stay) | Medicaid / Private"]
        out[f"{label} | Diff E(days) overall"] = b["Diff E(days) overall | Medicaid - Private"]
    return out

s4_point = s4_block(d18)
s4_boot = bootstrap(s4_block, d18, B_S4, label="Primary vs S4 (paired)")
s4_tab = summarise(s4_point, s4_boot)

ev_rows = []
for label in ["Primary", "S4"]:
    for dest in DC_LEVELS:
        r = s4_tab.set_index("estimate").loc[f"{label} | RR({dest})"]
        ev_rows.append({"model": label, "destination": dest, "RR_medicaid_vs_private": r.value,
                        "rr_ci_low": r.ci_low, "rr_ci_high": r.ci_high,
                        "E_value_point": evalue(r.value), "E_value_ci_bound": evalue_bound(r.ci_low, r.ci_high)})
ev_tab = pd.DataFrame(ev_rows)
print(s4_tab.round(4).to_string(index=False))
ev_tab.round(3)

In [ ]:
# ---- Export results only -------------------------------------------------------------
meta = pd.DataFrame({"item": ["run_date", "statsmodels", "seed", "B_S4", "n_18_64", "min_cell"],
                     "value": [pd.Timestamp.now().isoformat(), statsmodels.__version__, SEED, B_S4, len(d18), MIN_CELL]})
with pd.ExcelWriter(OUTFILE) as xw:
    meta.to_excel(xw, sheet_name="run_info", index=False)
    prev_tab.reset_index().to_excel(xw, sheet_name="s4_flag_prevalence", index=False)
    s4_tab.to_excel(xw, sheet_name="primary_vs_s4", index=False)
    ev_tab.to_excel(xw, sheet_name="evalues", index=False)
print("wrote", OUTFILE)
try:
    from google.colab import files
    files.download(OUTFILE)
except ImportError:
    pass